In [1]:
import pandas as pd
import os
import re
import numpy as np
from pathlib import Path

In [2]:
SSP_MODELING = Path.cwd().resolve().parents[2]
TABLEAU_DATA = SSP_MODELING / "tableau" / "data"
SCRIPT_DIR_PATH = os.getcwd()
CB_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(CB_DIR_PATH)
TORNADO_DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
INPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "input")
OUTPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "output")


In [3]:

TABLEAU_DATA

PosixPath('/Users/alexa/Projects/ssp_libya/ssp_modeling/tableau/data')

In [4]:
def add_sector_and_transformation_fields(df: pd.DataFrame, strategy_col: str = "strategy") -> pd.DataFrame:
    """
    Creates:
      - sector: sector code (e.g., AGRC)
      - transformation_name: transformation text after '... - <SECTOR>:'
    """
    df = df.copy()

    # --- sector extraction (captures 3-6 uppercase letters before colon) ---
    # Example: "Singleton - Default Value - AGRC: Improve rice..." -> AGRC
    df["sector"] = df[strategy_col].str.extract(r"-\s*([A-Z]{3,6})\s*:", expand=False)

    # Special case: baseline strategy
    df.loc[df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False), "sector"] = "BASE"

    # --- transformation_name extraction ---
    # Keep only text after "<SECTOR>:"
    # Example -> "Improve rice..."
    df["transformation_name"] = df[strategy_col].str.extract(r":\s*(.*)$", expand=False)

    # If it's baseline, keep the full strategy string as the name (or label it as BASE)
    base_mask = df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False)
    df.loc[base_mask, "transformation_name"] = "BASE"

    # Clean whitespace
    df["transformation_name"] = df["transformation_name"].fillna("").str.strip()

    return df

In [5]:
INPUT_DATA_DIR_PATH

'/Users/alexa/Projects/ssp_libya/ssp_modeling/cost-benefits/tornado_plot/scr/data/input'

## Load and process emission data

In [6]:
# Load the decomposed emissions long format data
emissions_df = pd.read_csv(os.path.join(TABLEAU_DATA, "decomposed_emissions_libya_2023.csv"))
emissions_df.head()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
0,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.050945,2023,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE,0.050945,NaN
1,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.024442,2024,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE,0.024442,NaN
2,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.024627,2025,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE,0.024627,NaN
3,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.024808,2026,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE,0.024808,NaN
4,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.024991,2027,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE,0.024991,NaN


In [7]:
print(emissions_df.primary_id.nunique())

4


In [8]:
print(emissions_df.primary_id.nunique())

4


In [9]:
# check unique strategy
emissions_df['strategy'].unique()

array(['Strategy TX:BASE', 'BAU', 'Unconditional', 'Conditional',
       'Historical'], dtype=object)

In [10]:
# Drop historical and tx:base from df
filtered_emissions_df = emissions_df.loc[~emissions_df['strategy'].isin(['Historical', 'Strategy TX:BASE'])]
print(emissions_df['strategy'].nunique())
print(filtered_emissions_df['strategy'].nunique())

5
3


In [11]:
filtered_emissions_df["strategy"].unique()

array(['BAU', 'Unconditional', 'Conditional'], dtype=object)

In [12]:
filtered_emissions_df.tail()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
4699,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2046,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE,0.0,NaN
4700,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2047,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE,0.0,NaN
4701,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2048,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE,0.0,NaN
4702,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2049,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE,0.0,NaN
4703,6005.0,76076.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.0,2050,N2O,0.0,0.0,Conditional,LBY,libya,SISEPUEDE,0.0,NaN


In [13]:
# Now concat the original base df and the filtered emissions df
tornado_emissions_df = filtered_emissions_df
tornado_emissions_df['strategy'].unique()

array(['BAU', 'Unconditional', 'Conditional'], dtype=object)

In [14]:
tornado_emissions_df.head()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
1176,6003.0,74074.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.050945,2023,CH4,0.0,0.0,BAU,LBY,libya,SISEPUEDE,0.050945,NaN
1177,6003.0,74074.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.024442,2024,CH4,0.0,0.0,BAU,LBY,libya,SISEPUEDE,0.024442,NaN
1178,6003.0,74074.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.024627,2025,CH4,0.0,0.0,BAU,LBY,libya,SISEPUEDE,0.024627,NaN
1179,6003.0,74074.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.024808,2026,CH4,0.0,0.0,BAU,LBY,libya,SISEPUEDE,0.024808,NaN
1180,6003.0,74074.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.024991,2027,CH4,0.0,0.0,BAU,LBY,libya,SISEPUEDE,0.024991,NaN


In [15]:
tornado_emissions_df.loc[
    tornado_emissions_df["subsector"].str.startswith("2."), "subsector"
] = "2 - IPPU"

In [16]:
# Aggregate by strategy_id, primary_id and strategy, and sum value
tornado_emissions_agg_df = tornado_emissions_df.groupby(
    ['strategy_id', 'primary_id', 'strategy', 'sector', 'subsector']
)['value'].sum().reset_index()

tornado_emissions_agg_df.head()


,strategy_id,primary_id,strategy,sector,subsector,value
0,6003.0,74074.0,BAU,1 - Energy,1.A.1 - Energy Industries,1113.863123
1,6003.0,74074.0,BAU,1 - Energy,1.A.2 - Manufacturing Industries and Construction,58.966287
2,6003.0,74074.0,BAU,1 - Energy,1.A.3 - Transport,676.890490
3,6003.0,74074.0,BAU,1 - Energy,1.A.4 - Other Sectors,25.312214
4,6003.0,74074.0,BAU,1 - Energy,1.B - Fugitive emissions from fuels,793.496337


In [17]:
tornado_emissions_agg_df.tail()

,strategy_id,primary_id,strategy,sector,subsector,value
31,6005.0,76076.0,Conditional,"3 - Agriculture, Forestry, and Other Land Use",3.B - Land,-33.483419
32,6005.0,76076.0,Conditional,"3 - Agriculture, Forestry, and Other Land Use",3.C - Aggregate sources and non-CO2 emissions ...,23.029126
33,6005.0,76076.0,Conditional,4 - Waste,4.A - Solid Waste Disposal,14.655026
34,6005.0,76076.0,Conditional,4 - Waste,4.D - Wastewater Treatment and Discharge,16.850836
35,6005.0,76076.0,Conditional,5- CCSQ,5- CCSQ,0.000000


In [18]:
tornado_emissions_agg_df['subsector'].unique()

array(['1.A.1 - Energy Industries',
       '1.A.2 - Manufacturing Industries and Construction',
       '1.A.3 - Transport', '1.A.4 - Other Sectors',
       '1.B - Fugitive emissions from fuels', '2 - IPPU',
       '3.A - Livestock', '3.B - Land',
       '3.C - Aggregate sources and non-CO2 emissions sources on land',
       '4.A - Solid Waste Disposal',
       '4.D - Wastewater Treatment and Discharge', '5- CCSQ'],
      dtype=object)

In [19]:
# check if strategy id nunique matches amount of rows
print(tornado_emissions_agg_df['strategy_id'].nunique())
print(tornado_emissions_agg_df.shape[0])

3
36


In [20]:
# rename value to emission_total
tornado_emissions_agg_df = tornado_emissions_agg_df.rename(columns={'value': 'emission_total'})

# create base_emission_total per subsector using strategy_id == 6003
base_by_subsector = tornado_emissions_agg_df.loc[
    tornado_emissions_agg_df['strategy_id'] == 6003, ['subsector', 'emission_total']
].rename(columns={'emission_total': 'base_emission_total'})

tornado_emissions_agg_df = tornado_emissions_agg_df.merge(base_by_subsector, on='subsector', how='left')

# calculate emission difference column
tornado_emissions_agg_df['emission_diff'] = tornado_emissions_agg_df['emission_total'] - tornado_emissions_agg_df['base_emission_total']

tornado_emissions_agg_df['emission_diff'] = tornado_emissions_agg_df['emission_diff'].round(1)

tornado_emissions_agg_df.head(40)

,strategy_id,primary_id,strategy,sector,subsector,emission_total,base_emission_total,emission_diff
0,6003.0,74074.0,BAU,1 - Energy,1.A.1 - Energy Industries,1113.863123,1113.863123,0.0
1,6003.0,74074.0,BAU,1 - Energy,1.A.2 - Manufacturing Industries and Construction,58.966287,58.966287,0.0
2,6003.0,74074.0,BAU,1 - Energy,1.A.3 - Transport,676.890490,676.890490,0.0
3,6003.0,74074.0,BAU,1 - Energy,1.A.4 - Other Sectors,25.312214,25.312214,0.0
4,6003.0,74074.0,BAU,1 - Energy,1.B - Fugitive emissions from fuels,793.496337,793.496337,0.0
5,6003.0,74074.0,BAU,2 - Industrial Processes and Product Use,2 - IPPU,209.195632,209.195632,0.0
6,6003.0,74074.0,BAU,"3 - Agriculture, Forestry, and Other Land Use",3.A - Livestock,32.946742,32.946742,0.0
7,6003.0,74074.0,BAU,"3 - Agriculture, Forestry, and Other Land Use",3.B - Land,-28.554219,-28.554219,0.0
8,6003.0,74074.0,BAU,"3 - Agriculture, Forestry, and Other Land Use",3.C - Aggregate sources and non-CO2 emissions ...,20.082066,20.082066,0.0
9,6003.0,74074.0,BAU,4 - Waste,4.A - Solid Waste Disposal,18.343463,18.343463,0.0


In [21]:
tornado_emissions_agg_df['strategy'].unique()

array(['BAU', 'Unconditional', 'Conditional'], dtype=object)

In [22]:
tornado_emissions_agg_df = tornado_emissions_agg_df[tornado_emissions_agg_df['strategy']!='NDC'] 

In [23]:
tornado_emissions_agg_df['strategy'].unique()

array(['BAU', 'Unconditional', 'Conditional'], dtype=object)

## Load and process CB data

In [24]:
# cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-01-29T15;28;40.322709_tornado_raw.csv"))
cb_raw_df = pd.read_csv(os.path.join(TABLEAU_DATA, "cb_data.csv"))
cb_raw_df.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,...,sector,cb_type,item_1,item_2,Year,strategy,strategy_id,primary_id,ids,gdp_mmm_usd
0,BASE,0,libya,8,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2023,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,101.162039
1,BASE,0,libya,9,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2024,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,105.576742
2,BASE,0,libya,10,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2025,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,107.688276
3,BASE,0,libya,11,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2026,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,109.842042
4,BASE,0,libya,12,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2027,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,112.038883


In [25]:
# --- Create a copy of the raw data ---
cb_data = cb_raw_df.copy()

cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,...,sector,cb_type,item_1,item_2,Year,strategy,strategy_id,primary_id,ids,gdp_mmm_usd
0,BASE,0,libya,8,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2023,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,101.162039
1,BASE,0,libya,9,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2024,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,105.576742
2,BASE,0,libya,10,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2025,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,107.688276
3,BASE,0,libya,11,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2026,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,109.842042
4,BASE,0,libya,12,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2027,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,112.038883


In [26]:
cb_data["sector"].unique()

array(['agrc', 'ccsq', 'entc', 'inen', 'ippu', 'lndu', 'lsmm', 'lvst',
       'scoe', 'soil', 'trns', 'trww', 'wali', 'waso'], dtype=object)

In [27]:
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,...,sector,cb_type,item_1,item_2,Year,strategy,strategy_id,primary_id,ids,gdp_mmm_usd
0,BASE,0,libya,8,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2023,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,101.162039
1,BASE,0,libya,9,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2024,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,105.576742
2,BASE,0,libya,10,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2025,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,107.688276
3,BASE,0,libya,11,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2026,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,109.842042
4,BASE,0,libya,12,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,...,agrc,crop_value,crops_produced,bevs_and_spices,2027,BASE,0,0,cb:agrc:crop_value:crops_produced:bevs_and_spi...,112.038883


In [28]:
cb_data['cb_type'].unique()

array(['crop_value', 'sector_specific', 'air_pollution', 'technical_cost',
       'ippu_value', 'ecosystem_services', 'fuel_cost', 'lvst_value',
       'land_pollution', 'congestion', 'road_safety', 'system_cost',
       'water_pollution', 'human_health', 'env_pollution',
       'technical_savings', 'consumer_savings'], dtype=object)

In [29]:
cb_data = cb_data[cb_data['cb_type']=='technical_cost']

In [30]:
cb_data['cb_type'].unique()

array(['technical_cost'], dtype=object)

In [31]:
cb_data['strategy_code'].unique()

array(['BASE', 'PFLO:UNCONDITIONAL', 'PFLO:CONDITIONAL'], dtype=object)

In [32]:
cb_data['strategy_code'].unique()

array(['BASE', 'PFLO:UNCONDITIONAL', 'PFLO:CONDITIONAL'], dtype=object)

In [33]:
cb_data = cb_data[cb_data['strategy_code']!='BASE']

In [34]:
cb_data['strategy_code'].unique()

array(['PFLO:UNCONDITIONAL', 'PFLO:CONDITIONAL'], dtype=object)

In [35]:
tornado_emissions_agg_df['subsector'].unique()

array(['1.A.1 - Energy Industries',
       '1.A.2 - Manufacturing Industries and Construction',
       '1.A.3 - Transport', '1.A.4 - Other Sectors',
       '1.B - Fugitive emissions from fuels', '2 - IPPU',
       '3.A - Livestock', '3.B - Land',
       '3.C - Aggregate sources and non-CO2 emissions sources on land',
       '4.A - Solid Waste Disposal',
       '4.D - Wastewater Treatment and Discharge', '5- CCSQ'],
      dtype=object)

In [36]:
SECTOR_TO_SUBSECTOR = {
    
    "entc": "1.A.1 - Energy Industries",
    "inen": "1.A.2 - Manufacturing Industries and Construction",
    "trns": "1.A.3 - Transport",
    "scoe": "1.A.4 - Other Sectors",

    'ippu': '2 - IPPU',

    "fgtv": "1.B - Fugitive emissions from fuels",

    "lvst": "3.A - Livestock",
    "lsmm": "3.A - Livestock",

    "soil": "3.C - Aggregate sources and non-CO2 emissions sources on land",
    "agr": "3.C - Aggregate sources and non-CO2 emissions sources on land",

    "waso": "4.A - Solid Waste Disposal",
    "trww": "4.D - Wastewater Treatment and Discharge",

    "ccsq": "5- CCSQ",

}

def add_subsector(df: pd.DataFrame, sector_col: str = "sector") -> pd.DataFrame:
    df = df.copy()
    df["subsector_ssp"] = df[sector_col].str.lower().map(SECTOR_TO_SUBSECTOR)
    return df

cb_data = add_subsector(cb_data)

In [37]:
# aggregate sum(value) grouped by strategy_code and sector
cb_data = (
    cb_data.groupby(["strategy_code", "strategy_id", "primary_id", "subsector_ssp"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "cumulative"})
)
cb_data.head(20)

,strategy_code,strategy_id,primary_id,subsector_ssp,cumulative
0,PFLO:CONDITIONAL,6005,76076,1.A.1 - Energy Industries,-3.796863e+01
1,PFLO:CONDITIONAL,6005,76076,1.A.2 - Manufacturing Industries and Construction,-1.078957e+01
2,PFLO:CONDITIONAL,6005,76076,1.A.3 - Transport,-3.681281e+00
3,PFLO:CONDITIONAL,6005,76076,1.A.4 - Other Sectors,-1.980725e+00
4,PFLO:CONDITIONAL,6005,76076,2 - IPPU,-5.989410e-08
5,PFLO:CONDITIONAL,6005,76076,3.C - Aggregate sources and non-CO2 emissions ...,6.495287e-04
6,PFLO:CONDITIONAL,6005,76076,4.A - Solid Waste Disposal,-2.588306e+00
7,PFLO:CONDITIONAL,6005,76076,4.D - Wastewater Treatment and Discharge,1.356040e+01
8,PFLO:UNCONDITIONAL,6004,75075,1.A.1 - Energy Industries,-1.921625e+01
9,PFLO:UNCONDITIONAL,6004,75075,1.A.2 - Manufacturing Industries and Construction,-7.842420e-01


In [38]:
tornado_emissions_agg_df.head()

,strategy_id,primary_id,strategy,sector,subsector,emission_total,base_emission_total,emission_diff
0,6003.0,74074.0,BAU,1 - Energy,1.A.1 - Energy Industries,1113.863123,1113.863123,0.0
1,6003.0,74074.0,BAU,1 - Energy,1.A.2 - Manufacturing Industries and Construction,58.966287,58.966287,0.0
2,6003.0,74074.0,BAU,1 - Energy,1.A.3 - Transport,676.890490,676.890490,0.0
3,6003.0,74074.0,BAU,1 - Energy,1.A.4 - Other Sectors,25.312214,25.312214,0.0
4,6003.0,74074.0,BAU,1 - Energy,1.B - Fugitive emissions from fuels,793.496337,793.496337,0.0


In [39]:
merged_df = tornado_emissions_agg_df.merge(
    cb_data[["strategy_id", "primary_id", "subsector_ssp", "cumulative"]],
    left_on=["strategy_id", "primary_id", "subsector"],
    right_on=["strategy_id", "primary_id", "subsector_ssp"],
    how="left"
).drop(columns="subsector_ssp")

merged_df.head(20)
  

,strategy_id,primary_id,strategy,sector,subsector,emission_total,base_emission_total,emission_diff,cumulative
0,6003.0,74074.0,BAU,1 - Energy,1.A.1 - Energy Industries,1113.863123,1113.863123,0.0,NaN
1,6003.0,74074.0,BAU,1 - Energy,1.A.2 - Manufacturing Industries and Construction,58.966287,58.966287,0.0,NaN
2,6003.0,74074.0,BAU,1 - Energy,1.A.3 - Transport,676.890490,676.890490,0.0,NaN
3,6003.0,74074.0,BAU,1 - Energy,1.A.4 - Other Sectors,25.312214,25.312214,0.0,NaN
4,6003.0,74074.0,BAU,1 - Energy,1.B - Fugitive emissions from fuels,793.496337,793.496337,0.0,NaN
5,6003.0,74074.0,BAU,2 - Industrial Processes and Product Use,2 - IPPU,209.195632,209.195632,0.0,NaN
6,6003.0,74074.0,BAU,"3 - Agriculture, Forestry, and Other Land Use",3.A - Livestock,32.946742,32.946742,0.0,NaN
7,6003.0,74074.0,BAU,"3 - Agriculture, Forestry, and Other Land Use",3.B - Land,-28.554219,-28.554219,0.0,NaN
8,6003.0,74074.0,BAU,"3 - Agriculture, Forestry, and Other Land Use",3.C - Aggregate sources and non-CO2 emissions ...,20.082066,20.082066,0.0,NaN
9,6003.0,74074.0,BAU,4 - Waste,4.A - Solid Waste Disposal,18.343463,18.343463,0.0,NaN


In [40]:
OUTPUT_DATA_DIR_PATH

'/Users/alexa/Projects/ssp_libya/ssp_modeling/cost-benefits/tornado_plot/scr/data/output'

In [42]:
merged_df.to_csv(os.path.join(TABLEAU_DATA, "mac_sector_libya.csv"), index=False)